In [28]:
import pandas as pd
import numpy as np

# ── 输入输出路径 ────────────────────────────────────────────
IN_PATH     = "target_match_dataset.csv"
SKILLS_PATH = "agent_team_skills.csv"
OUT_PATH    = "data/match_players_labeled.csv"

# ── Carrier 判定阈值（队内 z-score）───────────────────────
# 必要条件：ACS z ≥ 1.5（综合得分显著高于队友）
# 附加条件：ADR / 击杀 / KAST 至少一项 z ≥ 对应阈值
CARRIER_ACS_Z   = 1.5
CARRIER_ADR_Z   = 1.0
CARRIER_KILLS_Z = 1.0
CARRIER_KAST_Z  = 1.0

# ── Loafer 判定阈值 ────────────────────────────────────────
LOAFER_BOTTOM_N     = 2    # 每队 ACS 倒数几名进入候选
LOAFER_AFK_ROUNDS   = 2    # afk_rounds ≥ N 触发 flag_afk
LOAFER_SPAWN_ROUNDS = 5    # rounds_in_spawn ≥ N 触发 flag_afk
LOAFER_FF_DAMAGE    = 100  # ff_outgoing ≥ N 触发 flag_ff
LOAFER_KAST_Z       = -1.0 # KAST 代理 z ≤ N 触发 flag_kast
LOAFER_SKILLS_Z     = -1.0 # 团队技能率 z ≤ N 触发 flag_skills
LOAFER_ADR_Z        = -1.0 # ADR z ≤ N 触发 flag_damage
LOAFER_MIN_FLAGS    = 2    # 至少满足几个维度才标记为 loafer

## Step 3：Carrier 与 Loafer 识别

本 notebook 实现对 `target_match_dataset.csv` 中每名玩家的角色打标，输出 `match_players_labeled.csv`。

### 整体流程
```
原始数据
  └─ 1. 计算衍生指标（ACS / ADR / KAST代理 / 团队技能使用率）
       └─ 2. 识别 Carrier（队内显著突出的玩家）
            └─ 3. 识别 Loafer（ACS倒数预筛选 + 多维度打标）
                 └─ 4. 保存带标签的数据集
                      └─ 5. 校验 + 共存分析
```

## 1. 工具函数

**组内 z-score**：Valorant 中每队 5 人，玩家的绝对数值（比如 ACS=200）意义有限，
更重要的是「相对于自己队友」的表现差异。
因此所有指标都在 `match_id + team` 组内做标准化，再判定是否显著高/低。

> z = (自己的值 - 队伍均值) / 队伍标准差
>
> z > 0 表示高于队伍平均，z < 0 表示低于队伍平均

In [29]:
def zscore_within(df, col, group=("match_id", "team")):
    """
    在 group 指定的组内计算 z-score。
    
    若组内标准差为 0（所有人数值相同，比如全队技能使用都是 0），
    则填 0（避免除以零报错，也表示「无差异」）。
    """
    grp = df.groupby(list(group))[col]
    mu  = grp.transform("mean")
    sd  = grp.transform("std").replace(0, np.nan)
    return ((df[col] - mu) / sd).fillna(0)

## 2. 衍生指标计算

原始数据中只有累计值（总得分、总伤害等），需要换算成「每轮」的指标，方便跨对局比较（因为不同对局轮次数不同）。

| 指标 | 公式 | 含义 |
|------|------|------|
| **ACS** | score / rounds_played | 官方综合表现指标，越高越好 |
| **ADR** | damage_made / rounds_played | 每轮平均输出伤害 |
| **KAST 代理** | (kills + assists + survived_rounds) / rounds_played | 近似官方 KAST（Kill/Assist/Survive/Trade）；无法计算 Trade，用存活轮数近似 |
| **团队技能使用率** | 团队技能总施放次数 / rounds_played | 结合 `agent_team_skills.csv`，只统计该 agent 对团队有价值的技能 |

> 注：KAST 代理值可能 > 1（玩家在同一轮既击杀又存活），作为相对比较指标使用，不影响 z-score 计算。

In [30]:
def compute_metrics(df, skills_path=SKILLS_PATH):
    """
    在原始数据上添加所有派生指标列。
    
    团队技能使用率的计算逻辑：
      - 读取 agent_team_skills.csv，其中每个 agent 的 c/q/e/x_team 标注
        了哪些技能对团队有价值（1 = 团队技能，0 = 个人技能）
      - 将玩家的实际使用次数与该标注相乘，得到「团队技能施放总次数」
      - 除以 rounds_played 得到每轮使用率
      - 未在 CSV 中收录的新 agent 视为无团队技能（填 0）
    """
    rp = df["rounds_played"].replace(0, np.nan)  # 防止除以零

    # 基础绩效指标
    df["acs"]        = df["score"]       / rp
    df["adr"]        = df["damage_made"] / rp
    df["afk_rate"]   = df["afk_rounds"]  / rp

    # KAST 代理：(击杀 + 助攻 + 存活轮数) / 总轮数
    survived = (rp - df["deaths"]).clip(lower=0)
    df["kast_proxy"] = (df["kills"] + df["assists"] + survived) / rp

    # 合并 agent 团队技能标注
    skills = pd.read_csv(skills_path)
    df = df.merge(skills, on="agent", how="left")
    for col in ["c_team", "q_team", "e_team", "x_team"]:
        df[col] = df[col].fillna(0)  # 未知 agent → 无团队技能

    # 团队技能施放次数 = 各技能实际使用次数 × 是否为团队技能
    df["team_skill_casts"] = (
        df["c_cast"] * df["c_team"] +
        df["q_cast"] * df["q_team"] +
        df["e_cast"] * df["e_team"] +
        df["x_cast"] * df["x_team"]
    )
    df["team_skill_slots"] = df[["c_team","q_team","e_team","x_team"]].sum(axis=1)
    df["team_skill_rate"]  = df["team_skill_casts"] / rp  # 每轮团队技能使用次数

    # 清理合并时产生的辅助列
    df = df.drop(columns=["c_team","q_team","e_team","x_team"])
    return df

## 3. Carrier 识别

**定义**：在己方 5 人队内，表现显著突出的玩家。

**判定逻辑**（组内 z-score 比较）：

```
ACS z ≥ 1.5                ← 必要条件（综合得分显著高）
  AND
以下至少满足一项：
  ADR   z ≥ 1.0            ← 每轮输出伤害显著高
  击杀数 z ≥ 1.0            ← 击杀数显著高
  KAST  z ≥ 1.0            ← 团队贡献度显著高
```

> **为什么需要附加条件？**
> ACS 包含了分辅助分、首杀分等多个来源，单纯 ACS 高有时是因为打了很多辅助。
> 附加一个实战维度（伤害/击杀/KAST）可以避免误标纯辅助位为 carrier。

In [31]:
def label_carriers(df):
    """
    在 df 中新增以下列：
      z_acs, z_adr, z_kills, z_kast  — 各指标的队内 z-score
      is_carrier                     — 1 = carrier，0 = 普通
    """
    group = ("match_id", "team")

    df["z_acs"]   = zscore_within(df, "acs",        group)
    df["z_adr"]   = zscore_within(df, "adr",        group)
    df["z_kills"] = zscore_within(df, "kills",      group)
    df["z_kast"]  = zscore_within(df, "kast_proxy", group)

    # ACS 显著高（必要条件）AND 至少一个其他维度也显著高（附加条件）
    df["is_carrier"] = (
        (df["z_acs"]   >= CARRIER_ACS_Z) &
        (
            (df["z_adr"]   >= CARRIER_ADR_Z)  |
            (df["z_kills"] >= CARRIER_KILLS_Z) |
            (df["z_kast"]  >= CARRIER_KAST_Z)
        )
    ).astype(int)

    n = df["is_carrier"].sum()
    print(f"Carrier：{n} 人（占 {n/len(df):.1%}）")
    return df

## 4. Loafer 识别

**定义**：在团队中有明显「摆烂」或「社会懈怠」行为的玩家。

### 两步走设计

**步骤 1 — 预筛选（缩小候选范围）**
每队 5 人中 ACS 倒数 2 名进入候选。
理由：ACS 靠前的玩家（哪怕表现一般）很难定义为摆烂，先把他们排除。

**步骤 2 — 多维度打标（5 个维度，满足 ≥2 个 = loafer）**

| Flag | 触发条件 | 数据来源 |
|------|----------|----------|
| `flag_afk` | afk_rounds ≥ 2 或 rounds_in_spawn ≥ 5 | 最直观的不参与游戏行为 |
| `flag_ff` | ff_outgoing ≥ 100 | 大量伤害队友（主动捣乱）|
| `flag_kast` | KAST 代理 z ≤ -1.0 | 几乎不为团队做贡献 |
| `flag_skills` | 团队技能使用率 z ≤ -1.0（且 agent ≥ 2 个团队技能槽）| 不履行 agent 的团队职责 |
| `flag_damage` | ADR z ≤ -1.0 | 输出伤害显著低于队友 |

> `flag_skills` 附加了「agent ≥ 2 个团队技能槽」的条件，
> 是因为纯决斗位（如 Reyna/Jett）本身团队技能少，用绝对使用率不公平。

In [32]:
def label_loafers(df):
    """
    在 df 中新增以下列：
      acs_rank      — 队内 ACS 排名（1 = 最低）
      is_bottom_acs — 是否在 ACS 倒数 LOAFER_BOTTOM_N 名内
      flag_*        — 各维度摆烂标志（0/1）
      loafing_flags — 总 flag 数（0~5）
      is_loafer     — 1 = 高度疑似摆烂，0 = 正常
    """
    group = ("match_id", "team")

    # ── 步骤 1：ACS 倒数预筛选 ─────────────────────────────
    # rank 从 1 开始，1 = 队内 ACS 最低
    df["acs_rank"]      = df.groupby(list(group))["acs"].rank(method="min", ascending=True)
    df["is_bottom_acs"] = (df["acs_rank"] <= LOAFER_BOTTOM_N).astype(int)

    # ── 步骤 2：补算 z-score（若 label_carriers 未算过则补算）
    if "z_adr"  not in df.columns: df["z_adr"]  = zscore_within(df, "adr",        group)
    if "z_kast" not in df.columns: df["z_kast"] = zscore_within(df, "kast_proxy", group)
    df["z_skills"] = zscore_within(df, "team_skill_rate", group)

    # ── 步骤 3：各维度 flag ────────────────────────────────

    # Flag 1: 明显不参与游戏（挂机 / 长时间蹲 spawn）
    df["flag_afk"] = (
        (df["afk_rounds"]      >= LOAFER_AFK_ROUNDS)   |
        (df["rounds_in_spawn"] >= LOAFER_SPAWN_ROUNDS)
    ).astype(int)

    # Flag 2: 大量伤害队友（主动摆烂 / 捣乱）
    df["flag_ff"] = (df["ff_outgoing"] >= LOAFER_FF_DAMAGE).astype(int)

    # Flag 3: KAST 代理显著低（几乎不为团队做贡献）
    df["flag_kast"] = (df["z_kast"] <= LOAFER_KAST_Z).astype(int)

    # Flag 4: 团队技能使用率显著低
    # 附加条件：只对「应该使用团队技能」的 agent 判定（≥2 个团队技能槽）
    df["flag_skills"] = (
        (df["z_skills"]        <= LOAFER_SKILLS_Z) &
        (df["team_skill_slots"] >= 2)
    ).astype(int)

    # Flag 5: 伤害输出显著低（几乎不打伤害）
    df["flag_damage"] = (df["z_adr"] <= LOAFER_ADR_Z).astype(int)

    # ── 步骤 4：综合判定 ───────────────────────────────────
    flag_cols = ["flag_afk", "flag_ff", "flag_kast", "flag_skills", "flag_damage"]
    df["loafing_flags"] = df[flag_cols].sum(axis=1)

    # Loafer = 在 ACS 倒数候选名单内 AND 满足 ≥ LOAFER_MIN_FLAGS 个维度
    df["is_loafer"] = (
        (df["is_bottom_acs"]  == 1) &
        (df["loafing_flags"] >= LOAFER_MIN_FLAGS)
    ).astype(int)

    n = df["is_loafer"].sum()
    print(f"Loafer：{n} 人（占 {n/len(df):.1%}）")
    return df

## 5. 主流程

将以上函数串联起来：读取数据 → 计算指标 → 打标签 → 保存。

In [33]:
def build_labeled(in_path=IN_PATH, out_path=OUT_PATH):
    """
    主流程：
      输入：target_match_dataset.csv
      输出：data/match_players_labeled.csv（新增 carrier/loafer 标签）
    """
    print("读取原始数据...")
    df = pd.read_csv(in_path).drop_duplicates(subset=["match_id", "puuid"])
    print(f"  {df.shape[0]:,} 行 × {df.shape[1]} 列")

    print("\n计算衍生指标...")
    df = compute_metrics(df)

    print("\n识别 Carrier...")
    df = label_carriers(df)

    print("\n识别 Loafer...")
    df = label_loafers(df)

    df.to_csv(out_path, index=False)
    print(f"\n✅ 保存完成 → {out_path}  shape={df.shape}")
    return df

## 6. 合理性校验

打完标签后需要验证结果是否符合预期：

- **Carrier 比例**：Valorant 5v5 中 carrier 应该是少数（预期约 5%~15%）
- **Loafer 比例**：摆烂玩家也应是少数（预期约 5%~20%）
- **Carrier 与 Loafer 不应重叠**：同一玩家不可能同时是 carrier 和 loafer
- **Carrier 的各项指标均值应显著高于普通玩家**
- **Loafer 的各项指标均值应显著低于普通玩家**

In [34]:
def sanity_check(df):
    """对打标结果进行合理性校验，输出各项统计信息。"""
    n   = len(df)
    n_c = df["is_carrier"].sum()
    n_l = df["is_loafer"].sum()
    both = ((df["is_carrier"] == 1) & (df["is_loafer"] == 1)).sum()

    print("=" * 55)
    print("合理性校验")
    print("=" * 55)
    print(f"总行数：      {n:,}")
    print(f"Carrier：     {n_c}（{n_c/n:.1%}）")
    print(f"Loafer：      {n_l}（{n_l/n:.1%}）")
    print(f"两者重叠：    {both}（应为 0）")

    cols = ["acs", "adr", "kills", "kast_proxy"]
    print("\n── Carrier 均值 vs 普通玩家 ──")
    print(df.groupby("is_carrier")[cols].mean().round(1).to_string())

    print("\n── Loafer 均值 vs 普通玩家 ──")
    extra = ["afk_rate", "loafing_flags"]
    print(df.groupby("is_loafer")[cols + extra].mean().round(3).to_string())

    print("\n── 每队 Carrier 数量分布 ──")
    print(df.groupby(["match_id","team"])["is_carrier"].sum()
            .value_counts().sort_index().to_string())

    print("\n── 每队 Loafer 数量分布 ──")
    print(df.groupby(["match_id","team"])["is_loafer"].sum()
            .value_counts().sort_index().to_string())

    print("\n── loafing_flags 分布（0~5）──")
    print(df["loafing_flags"].value_counts().sort_index().to_string())
    print("=" * 55)

## 7. 共存分析

**核心研究问题**：队伍里有 carrier 时，其他队友的懈怠程度是否更高？

**方法**：
1. 计算每队是否存在 carrier（`team_has_carrier`）
2. 只看非 carrier 的队友（carrier 本人排除出分析）
3. 比较「有 carrier 的队伍」vs「无 carrier 的队伍」中，队友的 loafing_flags 和 is_loafer 比例

这是 Research Question 1 的初步答案。

In [35]:
def coexistence_check(df):
    """
    分析有/无 carrier 的队伍中，非 carrier 队友的懈怠指标对比。
    返回 teammates DataFrame（非 carrier 子集），供后续深入分析使用。
    """
    # 计算每队是否存在 carrier
    team_has_carrier = (
        df.groupby(["match_id", "team"])["is_carrier"]
          .max()
          .rename("team_has_carrier")
          .reset_index()
    )
    df = df.merge(team_has_carrier, on=["match_id", "team"])

    # 只分析非 carrier 的队友
    teammates = df[df["is_carrier"] == 0].copy()

    print("=" * 55)
    print("共存分析：有/无 Carrier 队伍中的非 Carrier 队友")
    print("=" * 55)

    print("\n── loafing_flags 均值（越高 = 懈怠迹象越多）──")
    print(teammates.groupby("team_has_carrier")["loafing_flags"].mean().round(3).to_string())

    print("\n── 各维度 flag 触发率 ──")
    flag_cols = ["flag_afk", "flag_ff", "flag_kast", "flag_skills", "flag_damage"]
    print(teammates.groupby("team_has_carrier")[flag_cols].mean().round(3).to_string())

    print("\n── is_loafer 比例 ──")
    print(teammates.groupby("team_has_carrier")["is_loafer"].mean().round(3).to_string())

    # 同时有 carrier 和 loafer 的队伍（「共存局」）
    coexist = (
        df.groupby(["match_id", "team"])
          .agg(n_carrier=("is_carrier","sum"), n_loafer=("is_loafer","sum"))
          .query("n_carrier >= 1 and n_loafer >= 1")
          .reset_index()
    )
    print(f"\n共存局（同队有 carrier + loafer）：{len(coexist)} 队")

    # 随机抽 3 个共存局展示明细（含各维度 flag）
    print("\n── 随机抽取 3 个共存局明细 ──")
    show = ["name", "acs", "kills", "afk_rounds",
            "flag_afk", "flag_ff", "flag_kast", "flag_skills", "flag_damage",
            "loafing_flags", "is_carrier", "is_loafer"]
    for _, row in coexist.sample(min(3, len(coexist)), random_state=42).iterrows():
        sub = df[(df["match_id"] == row["match_id"]) & (df["team"] == row["team"])]
        print(f"\nmatch={row['match_id'][:12]}  team={row['team']}")
        print(sub[show].to_string(index=False))

    return teammates

## 8. 执行

In [36]:
df = build_labeled()
sanity_check(df)
teammates = coexistence_check(df)

读取原始数据...
  14,810 行 × 37 列

计算衍生指标...

识别 Carrier...
Carrier：757 人（占 5.1%）

识别 Loafer...
Loafer：1317 人（占 8.9%）

✅ 保存完成 → data/match_players_labeled.csv  shape=(14810, 59)
合理性校验
总行数：      14,810
Carrier：     757（5.1%）
Loafer：      1317（8.9%）
两者重叠：    0（应为 0）

── Carrier 均值 vs 普通玩家 ──
              acs    adr  kills  kast_proxy
is_carrier                                 
0           205.8  135.8   15.4         1.2
1           324.3  206.5   24.6         1.7

── Loafer 均值 vs 普通玩家 ──
               acs      adr   kills  kast_proxy  afk_rate  loafing_flags
is_loafer                                                               
0          219.861  144.467  16.514       1.296       0.0          0.170
1          130.163   87.741   9.506       0.837       0.0          2.046

── 每队 Carrier 数量分布 ──
is_carrier
0    2205
1     757

── 每队 Loafer 数量分布 ──
is_loafer
0    1648
1    1311
2       3

── loafing_flags 分布（0~5）──
loafing_flags
0    11219
1     2250
2     1279
3       62
共存分析：有/无 Carrier 队伍中

In [37]:
df.head()

,match_id,puuid,name,tag,team,region,map,patch,rounds_played,game_start,...,acs_rank,is_bottom_acs,z_skills,flag_afk,flag_ff,flag_kast,flag_skills,flag_damage,loafing_flags,is_loafer
0,093b6ddb-15aa-4e23-8a14-d5f035ffc9b9,13f91964-c9cf-5880-9e6b-86edf482d3a5,secret base,1111,Blue,kr,Bind,release-12.06-shipping-19-4440219,22,1776004484,...,2.0,1,-1.061988,0,0,1,0,0,1,0
1,093b6ddb-15aa-4e23-8a14-d5f035ffc9b9,ee397c09-c873-5939-9a77-b89328f36f9a,HeadSh0tW,韓傘友,Blue,kr,Bind,release-12.06-shipping-19-4440219,22,1776004484,...,1.0,1,-0.212398,0,0,0,0,1,1,0
2,093b6ddb-15aa-4e23-8a14-d5f035ffc9b9,d201fcc3-780d-5a07-acc8-7f6075b1bf7c,퇴 물,0121,Red,kr,Bind,release-12.06-shipping-19-4440219,22,1776004484,...,1.0,1,-0.697139,0,0,1,0,1,2,1
3,093b6ddb-15aa-4e23-8a14-d5f035ffc9b9,c163a0da-b4c8-5235-9cad-a41ddd213a4b,민철갓,1111,Red,kr,Bind,release-12.06-shipping-19-4440219,22,1776004484,...,2.0,1,0.117276,0,0,0,0,0,0,0
4,093b6ddb-15aa-4e23-8a14-d5f035ffc9b9,cc577deb-338b-5a2c-b131-38aa38c4ed8c,아카츠키는위대하다,BERH,Blue,kr,Bind,release-12.06-shipping-19-4440219,22,1776004484,...,4.0,0,0.283197,0,0,0,0,0,0,0
